# 🚩 Sprint 1 · Checkpoint 02 — The Geometry

**Companion ao Boyd (Cap 2 — Convex Sets) + Weinberger (Lec 3-5).**

No CP01 trabalhaste em 1D. Real-world ML é multidimensional. Este checkpoint constrói a álgebra das matrizes (transposta, multiplicação, diagonal), a *paisagem do erro* (MSE), e a noção que governa toda a otimização moderna: **convexidade**.

**Como trabalhar:**
- Cada `# TODO` é uma micro-task. Substitui `...` ou completa o esqueleto.
- A célula corre uma `assert` — se falhar, lê a mensagem e corrige.
- Trabalha em `solutions/local.ipynb` (cópia local).

**Pergunta-âncora** (responde no fim): *qual é o teste-relâmpago para saberes se um conjunto é convexo?*

## ⚙️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

def check(cond, msg='falhou'):
    assert cond, f'❌ {msg}'
    print('✅', msg)

## §1 — Matrizes: a aritmética

**Weinberger Lec 3 (KNN), 4-5.**

Uma matriz é uma forma compacta de empacotar muitos vetores em paralelo. Em ML:

- Cada linha é uma **observação** (um cliente, um pixel, um dia de mercado).
- Cada coluna é uma **feature** (idade, intensidade, retorno).

A grande maioria das operações de ML reduz-se a três coisas: **transpor**, **multiplicar matrizes**, e **diagonalizar**. Vamos codar cada uma do zero antes de usar `np.dot` / `@`.

### Task 1.1 — Transposta

A transposta troca linhas com colunas: $(A^T)_{ij} = A_{ji}$.

Para uma matriz $m \times n$, o resultado é $n \times m$.

Implementa em **duas versões**:
1. Manual com `np.empty` + duplo loop.
2. Em uma linha usando `.T` ou `np.transpose`.

In [ ]:
def transpose_manual(A):
    """Transposta via loops explícitos."""
    A = np.asarray(A, dtype=float)
    m, n = A.shape
    out = np.empty((n, m), dtype=float)
    # TODO: dois loops aninhados, out[j, i] = A[i, j]
    return out

def transpose_numpy(A):
    """Transposta numpy (uma linha)."""
    # TODO: usa .T ou np.transpose
    return ...

A = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])
T_manual = transpose_manual(A)
T_numpy = transpose_numpy(A)

check(T_manual.shape == (3, 2), f'transposta de (2,3) deve ter shape (3,2), obteve {T_manual.shape}')
check(np.allclose(T_manual, T_numpy), 'manual e numpy devem coincidir')
check(T_manual[0, 0] == 1 and T_manual[2, 1] == 6, 'elementos transpostos corretos')

# Propriedade: (A^T)^T = A
check(np.allclose(transpose_numpy(transpose_numpy(A)), A),
      '(A^T)^T deve ser igual a A')

# Propriedade em quadrada: simetria iff A = A^T
S = np.array([[1, 2], [2, 1]], dtype=float)
check(np.allclose(transpose_numpy(S), S), 'matriz simétrica: A = A^T')

### Task 1.2 — Multiplicação de matrizes

A operação central. Para $A$ de shape $(m, k)$ e $B$ de shape $(k, n)$:

$$ (AB)_{ij} = \sum_{p=1}^{k} A_{ip} B_{pj} $$

O resultado tem shape $(m, n)$. As dimensões internas têm que coincidir (ambas $k$).

Implementa em duas versões:
1. **Manual** — triplo loop. Lento mas mostra a mecânica.
2. **NumPy** — `A @ B` ou `np.dot(A, B)`.

Compara performance num exemplo com matrizes 100×100 (deve ser ~100× mais rápido com NumPy).

In [ ]:
def matmul_manual(A, B):
    """Multiplicação via triplo loop."""
    A = np.asarray(A, dtype=float); B = np.asarray(B, dtype=float)
    m, k = A.shape
    k2, n = B.shape
    assert k == k2, f'dimensões internas têm que coincidir: A é {A.shape}, B é {B.shape}'
    C = np.zeros((m, n), dtype=float)
    # TODO: três loops aninhados (i sobre m, j sobre n, p sobre k)
    # TODO: C[i, j] += A[i, p] * B[p, j]
    return C

def matmul_numpy(A, B):
    # TODO: usa @ ou np.dot
    return ...

# Caso pequeno verificável à mão
A = np.array([[1, 2], [3, 4]], dtype=float)
B = np.array([[5, 6], [7, 8]], dtype=float)
expected = np.array([[19, 22], [43, 50]], dtype=float)  # 1·5+2·7=19, 1·6+2·8=22, etc.

check(np.allclose(matmul_manual(A, B), expected), 'manual: caso 2×2 conhecido')
check(np.allclose(matmul_numpy(A, B), expected), 'numpy: caso 2×2 conhecido')

# Identidade preserva: A · I = A
I = np.eye(2)
check(np.allclose(matmul_numpy(A, I), A), 'A · I deve ser igual a A')

# (AB)^T = B^T A^T — propriedade que vais usar mil vezes em backprop
AB = matmul_numpy(A, B)
check(np.allclose(AB.T, matmul_numpy(B.T, A.T)),
      '(AB)^T deve ser igual a B^T · A^T')

### Task 1.3 — Diagonal e identidade

A **diagonal** de uma matriz quadrada são os elementos $A_{ii}$. Aparece em:
- **Traço:** $\text{tr}(A) = \sum_i A_{ii}$ (soma da diagonal). Usado em ML em normas e provas.
- **Identidade:** matriz com 1s na diagonal e 0s fora. Funciona como o `1` da multiplicação matricial.

Implementa `diagonal_manual(A)` (devolve vetor) e `identity_manual(n)` (devolve matriz).

In [ ]:
def diagonal_manual(A):
    """Vetor com a diagonal de A."""
    A = np.asarray(A, dtype=float)
    n = min(A.shape)
    out = np.empty(n, dtype=float)
    # TODO: loop, out[i] = A[i, i]
    return out

def identity_manual(n):
    """Matriz identidade n×n."""
    out = np.zeros((n, n), dtype=float)
    # TODO: para i em range(n), out[i, i] = 1
    return out

A = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=float)
check(np.allclose(diagonal_manual(A), [1, 5, 9]), f'diagonal deve ser [1, 5, 9], obteve {diagonal_manual(A)}')

# Coerência com np.diag
check(np.allclose(diagonal_manual(A), np.diag(A)), 'diagonal_manual deve coincidir com np.diag')

I3 = identity_manual(3)
check(np.allclose(I3, np.eye(3)), 'identity_manual deve coincidir com np.eye')

# Traço: soma da diagonal
trace = float(np.sum(diagonal_manual(A)))
check(abs(trace - 15) < 1e-9, f'tr(A) = 1+5+9 = 15, obteve {trace}')

## §2 — Geometria do erro: MSE

A função-perda mais usada em regressão é o **Mean Squared Error**:

$$ \text{MSE}(y, \hat{y}) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 $$

Onde $y$ são as labels reais e $\hat{y}$ as predições do modelo. É *exatamente* esta função que vais minimizar nos próximos checkpoints com GD, Newton, Adam — porque é convexa e diferenciável.

**Por que MSE e não MAE?** A derivada do MSE é linear no erro (`2(ŷ-y)/n`), ideal para gradient descent. O MAE tem derivada descontínua em 0.

### Task 2.1 — MSE manual e vectorizado

Implementa em duas versões: loop e numpy. Para vetores grandes, a versão numpy deve ser muito mais rápida.

In [ ]:
def mse_manual(y, y_hat):
    y = np.asarray(y, dtype=float); y_hat = np.asarray(y_hat, dtype=float)
    assert y.shape == y_hat.shape
    n = len(y)
    total = 0.0
    # TODO: loop, total += (y[i] - y_hat[i]) ** 2
    return total / n

def mse_numpy(y, y_hat):
    y = np.asarray(y, dtype=float); y_hat = np.asarray(y_hat, dtype=float)
    # TODO: uma linha — usa np.mean e ** 2
    return ...

# Casos canónicos
y = np.array([1.0, 2.0, 3.0, 4.0])
y_hat = np.array([1.0, 2.0, 3.0, 4.0])
check(abs(mse_numpy(y, y_hat)) < 1e-9, 'predição perfeita → MSE = 0')

y_hat_bad = np.array([2.0, 3.0, 4.0, 5.0])  # cada erro = 1
check(abs(mse_numpy(y, y_hat_bad) - 1.0) < 1e-9, 'erro constante de 1 → MSE = 1')

# Manual e numpy coincidem
y_rand = rng.normal(size=100)
y_hat_rand = y_rand + rng.normal(scale=0.5, size=100)
check(abs(mse_manual(y_rand, y_hat_rand) - mse_numpy(y_rand, y_hat_rand)) < 1e-9,
      'manual e numpy devem coincidir')

# MSE é simétrico: mse(y, ŷ) = mse(ŷ, y)
check(abs(mse_numpy(y, y_hat_bad) - mse_numpy(y_hat_bad, y)) < 1e-9,
      'MSE é simétrico nos argumentos')

### Task 2.2 — A paisagem do erro em 2D

Considera o problema de regressão linear de uma feature: $\hat{y}_i = w \cdot x_i + b$.

Dados $(x_i, y_i)$ fixos, a função $L(w, b) = \text{MSE}(y, w x + b)$ é uma função de **duas variáveis**. A sua paisagem é um **parabolóide** (sempre convexo!).

Vamos visualizá-la para 30 pontos sintéticos onde a verdade é $y = 2x + 1$.

In [ ]:
# Dados sintéticos com verdade w=2, b=1
n_pts = 30
x_data = rng.uniform(-3, 3, size=n_pts)
y_data = 2.0 * x_data + 1.0 + rng.normal(scale=0.3, size=n_pts)

def loss_at(w, b):
    """L(w, b) = MSE(y_data, w · x_data + b)."""
    y_hat = w * x_data + b
    return mse_numpy(y_data, y_hat)

# Grid 2D
ws = np.linspace(0, 4, 80)
bs = np.linspace(-1, 3, 80)
W, B = np.meshgrid(ws, bs)
L = np.vectorize(loss_at)(W, B)

# Visualiza com contour
fig, ax = plt.subplots(figsize=(7, 5))
cs = ax.contour(W, B, L, levels=20, cmap='viridis')
ax.clabel(cs, inline=True, fontsize=8)
ax.plot(2.0, 1.0, 'r*', markersize=15, label='verdade (w=2, b=1)')
ax.set_xlabel('w (declive)'); ax.set_ylabel('b (intercepto)')
ax.set_title('Paisagem MSE — convexa (parabolóide)'); ax.legend()
plt.show()

# Verifica que o mínimo da grid está perto da verdade
i_min, j_min = np.unravel_index(np.argmin(L), L.shape)
w_argmin, b_argmin = W[i_min, j_min], B[i_min, j_min]
print(f'argmin na grid: w≈{w_argmin:.2f}, b≈{b_argmin:.2f}')
check(abs(w_argmin - 2.0) < 0.2 and abs(b_argmin - 1.0) < 0.3,
      f'argmin deve estar perto de (2, 1), obteve ({w_argmin:.2f}, {b_argmin:.2f})')

## §3 — Conjuntos convexos

**Boyd, Cap 2.1.**

> Um conjunto $C$ é **convexo** se, para quaisquer dois pontos $x, y \in C$ e qualquer $\theta \in [0, 1]$, o ponto $\theta x + (1-\theta) y$ também está em $C$.

Em prosa: *qualquer segmento de reta entre dois pontos do conjunto fica todo dentro do conjunto.*

Por que importa? Porque a otimização de uma função convexa sobre um conjunto convexo tem **garantias fortes**: o mínimo local é global, e os algoritmos do tipo GD/Newton convergem (com a `lr` certa). Otimização não-convexa é o mundo selvagem do deep learning, sem garantias.

### Task 3.1 — Combinação convexa

Dados dois pontos $x, y \in \mathbb{R}^d$ e $\theta \in [0, 1]$, devolve $\theta x + (1-\theta) y$.

Verifica:
- $\theta = 0$ devolve $y$
- $\theta = 1$ devolve $x$
- $\theta = 0.5$ devolve o ponto médio

In [ ]:
def convex_combination(x, y, theta):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    assert 0 <= theta <= 1, f'theta deve estar em [0,1], obteve {theta}'
    # TODO: devolve theta * x + (1 - theta) * y
    return ...

x = np.array([1.0, 0.0])
y = np.array([0.0, 1.0])

check(np.allclose(convex_combination(x, y, 1.0), x), 'theta=1 → x'),
check(np.allclose(convex_combination(x, y, 0.0), y), 'theta=0 → y'),
check(np.allclose(convex_combination(x, y, 0.5), [0.5, 0.5]), 'theta=0.5 → ponto médio'),

# Combinação convexa fica sempre na bola unitária se x, y já estiverem
for _ in range(10):
    th = rng.uniform(0, 1)
    p = convex_combination(x, y, th)
    check(np.linalg.norm(p) <= 1.0 + 1e-9,
          f'combinação convexa de pontos na bola fica na bola: theta={th:.2f}, ||p||={np.linalg.norm(p):.4f}')

### Task 3.2 — Verifica convexidade visualmente

Dois conjuntos em 2D:
- **Disco**: $\{(x, y) : x^2 + y^2 \le 1\}$ — convexo.
- **Anel**: $\{(x, y) : 0.3 \le \sqrt{x^2 + y^2} \le 1\}$ — **não-convexo** (o segmento entre dois pontos opostos passa pelo buraco central).

Implementa as funções de teste de pertença e visualiza.

In [ ]:
def in_disc(p, r=1.0):
    p = np.asarray(p, dtype=float)
    # TODO: True se ||p||² ≤ r², False caso contrário
    return ...

def in_ring(p, r_inner=0.3, r_outer=1.0):
    p = np.asarray(p, dtype=float)
    n = np.linalg.norm(p)
    # TODO: True se r_inner ≤ ||p|| ≤ r_outer
    return ...

# Disco: amostra muitos pontos da combinação convexa de dois pontos no disco
# e confirma que TODOS ficam no disco.
fails_disc = 0
for _ in range(200):
    a = rng.uniform(-1, 1, 2); a = a / max(np.linalg.norm(a), 1.0) * 0.9
    b = rng.uniform(-1, 1, 2); b = b / max(np.linalg.norm(b), 1.0) * 0.9
    th = rng.uniform(0, 1)
    if not in_disc(convex_combination(a, b, th)):
        fails_disc += 1
check(fails_disc == 0, f'disco é convexo: 0 falhas em 200, obteve {fails_disc}')

# Anel: encontra um par concreto (oposto) cuja combinação está fora.
a_ring = np.array([-0.7, 0.0])  # no anel
b_ring = np.array([0.7, 0.0])   # no anel
mid = convex_combination(a_ring, b_ring, 0.5)  # = (0,0), no buraco!
check(in_ring(a_ring) and in_ring(b_ring), 'a_ring e b_ring devem estar no anel')
check(not in_ring(mid),
      f'ponto médio entre (-0.7,0) e (0.7,0) deve estar FORA do anel (no buraco). mid={mid}, ||mid||={np.linalg.norm(mid):.3f}')

# Visualização rápida
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
theta_circle = np.linspace(0, 2*np.pi, 200)

# Disco
axes[0].fill(np.cos(theta_circle), np.sin(theta_circle), alpha=0.3, color='steelblue')
axes[0].plot([-0.7, 0.7], [0, 0], 'r-o', label='segmento — fica dentro')
axes[0].set_aspect('equal'); axes[0].set_title('Disco (convexo)'); axes[0].legend()
axes[0].set_xlim(-1.2, 1.2); axes[0].set_ylim(-1.2, 1.2)

# Anel
axes[1].fill(np.cos(theta_circle), np.sin(theta_circle), alpha=0.3, color='steelblue')
axes[1].fill(0.3*np.cos(theta_circle), 0.3*np.sin(theta_circle), color='white')
axes[1].plot([-0.7, 0.7], [0, 0], 'r-o', label='segmento — atravessa o buraco')
axes[1].set_aspect('equal'); axes[1].set_title('Anel (NÃO convexo)'); axes[1].legend()
axes[1].set_xlim(-1.2, 1.2); axes[1].set_ylim(-1.2, 1.2)
plt.tight_layout(); plt.show()

## 🏁 Geometry — fechado

Se chegaste aqui sem `AssertionError`, dominaste:

1. **Aritmética matricial** — transposta, multiplicação (manual + numpy), diagonal, traço.
2. **MSE como paisagem 2D** — viste que a regressão linear é minimização sobre uma função convexa em $(w, b)$.
3. **Conjuntos convexos** — combinação convexa, disco vs anel, e o teste-relâmpago de Boyd.

**Perguntas de fecho:**
- Por que é que `(AB)ᵀ = BᵀAᵀ` (e não `AᵀBᵀ`)?
- Em que diferem `AB` e `BA`? São sempre iguais? Quando comutam?
- Qual é o teste de uma linha para verificar se um conjunto é convexo?

**Próximo checkpoint:** [03 — Velocity](../03_Checkpoint_Velocity/) — pegamos no GD do CP01 e adicionamos momento, Nesterov, AdaGrad. Vais aplicar à regressão que visualizaste em §2.2 deste.